# Transformation and Load
Loads the cleaned data, adds derived metrics, and loads the result into PostgreSQL.

In [1]:
import pandas as pd

clean = pd.read_csv(
    "../data/processed/orders_cleaned.csv",
    parse_dates=["order_date"],
    dtype={"delivery_days": "Int64", "customer_rating": "Int64", "return_flag": "boolean"},
)
clean.shape

(5000, 24)

## Money columns

In [2]:
clean["gross_amount"] = clean["quantity"] * clean["unit_price"]
clean["discount_amount"] = clean["gross_amount"] * clean["discount_rate"]
clean["total_amount"] = clean["order_revenue"] + clean["delivery_fee"]

## Date columns

In [4]:
clean["order_year"] = clean["order_date"].dt.year
clean["order_month"] = clean["order_date"].dt.month
clean["order_quarter"] = clean["order_date"].dt.quarter
clean["order_day"] = clean["order_date"].dt.day_name()

## Yes/No flags

In [5]:
clean["is_completed"] = clean["order_status"] == "Completed"
clean["is_delayed"] = clean["delivery_status"] == "Delayed"
clean["delivery_speed"] = pd.cut(clean["delivery_days"].astype("float"),
                                 bins=[0, 3, 7, 14], labels=["Fast", "Standard", "Slow"])
clean["status_conflict"] = clean["order_status"].isin(["Cancelled", "Pending"]) & (clean["delivery_status"] == "Delivered")

## Check and save

In [6]:
print(clean.shape)
print("Completed:", clean["is_completed"].sum())
print("Delayed:", clean["is_delayed"].sum())
print("Status conflicts:", clean["status_conflict"].sum())

clean.to_csv("../data/processed/orders_transformed.csv", index=False)

(5000, 35)
Completed: 2252
Delayed: 589
Status conflicts: 798


# Load into PostgreSQL

### Installing psycopg2

In [7]:
%pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env")
engine = create_engine(os.getenv("DATABASE_URL"))

In [9]:
clean.to_sql("orders", engine, if_exists="replace", index=False)

330

### Checking that no data was lost

In [10]:
check = pd.read_sql("SELECT COUNT(*) AS rows, SUM(order_revenue) AS revenue FROM orders", engine)
print(check)
print("Pandas revenue:", clean["order_revenue"].sum())

   rows      revenue
0  5000  452887325.0
Pandas revenue: 452887325.0
